# DGMAE Figure 8 reproduction
Train the encoders on Roman, Squirrel and Actor, then measure cosine
similarity on unique cross-label edges. The comparison uses seed 0 and
the protocol documented in the experiment README.
Smoke trains each model for one epoch. Full uses the configured pretrain
budgets. Both save PNG, PDF, raw cosine scores, and metadata.


In [ ]:
MODE = "configure"  # "configure", "smoke" or "full"
DATASETS = ["roman", "squirrel", "actor"]
SEED = 0


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

assert MODE in {"configure", "smoke", "full"}
work = Path("/tmp/dgmae-figure8")
checkout = work / "source"
graphmae = work / "graphmae"
greet = work / "greet"
experiment_commit = "b5752583ba117b886c9cd9ac544c49bccd3f3666"
graphmae_commit = "b14f080c919257b495e3cb6474286d5252d6a635"
greet_commit = "8bcc94090fe5f343aaeedcda981c5344d496a7f1"

def clone(url, commit, destination):
    destination.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q", str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "fetch", "--depth=1", url, commit], check=True)
    subprocess.run(["git", "-C", str(destination), "checkout", "--detach", commit], check=True)

if MODE != "configure":
    subprocess.run(["nvidia-smi", "-L"], check=True)
    clone("https://github.com/phud774/CS313.R11.git", experiment_commit, checkout)
    clone("https://github.com/THUDM/GraphMAE.git", graphmae_commit, graphmae)
    clone("https://github.com/yixinliu233/GREET.git", greet_commit, greet)
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.13.7"], check=True)
    setup_dir = checkout / "experiments" / "ablation"
    subprocess.run(["bash", str(setup_dir / "setup_kaggle.sh"), str(work)], check=True)
    subprocess.run(["uv", "pip", "install", "--python", str(work / ".dgmae-venv/bin/python"),
                    "matplotlib==3.9.4", "tensorboardX==2.6.2.2"], check=True)


In [ ]:
if MODE != "configure":
    from IPython.display import Image, display

    script = checkout / "DGMAE-REIMPL/experiments/figure8/run_figure8.py"
    output = Path("/kaggle/working/dgmae-figure8") / MODE
    cmd = [
        "bash", str(setup_dir / "run_kaggle_python.sh"), str(work), "-B", str(script),
        "--graphmae", str(graphmae), "--greet", str(greet), "--data", str(work / "datasets"),
        "--output", str(output), "--datasets", *DATASETS, "--seed", str(SEED),
    ]
    if MODE == "smoke":
        cmd.append("--smoke")
    env = dict(os.environ, PYTHONUNBUFFERED="1", DGLBACKEND="pytorch", OMP_NUM_THREADS="4")
    subprocess.run(cmd, cwd=greet, env=env, check=True)
    display(Image(filename=str(output / "figure8.png")))
